# Homogeneização com PINNs — versão GPU (CUDA) no Google Colab

Notebook para rodar a pasta `Homogen_CUDA` em uma GPU do Colab.

**IMPORTANTE — ative a GPU antes de começar:**
`Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **T4 GPU**.

Execute as células na ordem. A primeira execução precompila Julia + CUDA + Zygote e leva alguns minutos (normal).

## 1. Confere a GPU

In [ ]:
!nvidia-smi

## 2. Instala o Julia
Usamos o Julia 1.10 (LTS), bem estável com o CUDA.jl.

In [ ]:
%%bash
set -e
JULIA_VER=1.10.5
if [ ! -x /usr/local/bin/julia ]; then
  wget -q https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-${JULIA_VER}-linux-x86_64.tar.gz
  tar -xzf julia-${JULIA_VER}-linux-x86_64.tar.gz
  ln -sf $PWD/julia-${JULIA_VER}/bin/julia /usr/local/bin/julia
fi
julia --version

## 3. Envia o código para o Colab

**Opção A (recomendada): faça upload de um .zip da pasta `Homogen_CUDA`.**
Compacte a pasta no seu computador (botão direito → "Compactar") e rode a célula abaixo para enviar o arquivo. O nome do zip pode ser qualquer um — ele será descompactado e a pasta `Homogen_CUDA` deve aparecer.

Se preferir, use a **Opção B** (Google Drive) na célula seguinte.

In [ ]:
# Opção A — upload de um .zip contendo a pasta Homogen_CUDA
from google.colab import files
import zipfile, os, glob

up = files.upload()                     # escolha o .zip
zip_name = list(up.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')

# Localiza a pasta que contem o main.jl
main_path = glob.glob('**/Homogen_CUDA/main.jl', recursive=True) or glob.glob('**/main.jl', recursive=True)
assert main_path, 'main.jl nao encontrado no zip'
PROJ_DIR = os.path.dirname(os.path.abspath(main_path[0]))
print('Pasta do projeto:', PROJ_DIR)

In [ ]:
# Opção B — (alternativa) montar o Google Drive e apontar a pasta
# from google.colab import drive
# drive.mount('/content/drive')
# PROJ_DIR = '/content/drive/MyDrive/Homogen_CUDA'   # ajuste o caminho
# print('Pasta do projeto:', PROJ_DIR)

## 4. Instala os pacotes Julia
Cria um ambiente dentro da pasta do projeto e adiciona as dependências. Só precisa rodar uma vez por sessão.

In [ ]:
import os
os.environ['PROJ_DIR'] = PROJ_DIR

In [ ]:
%%bash
cd "$PROJ_DIR"
julia --project=. -e '
  using Pkg
  Pkg.add(["CUDA", "Zygote", "Sobol", "Plots", "DelimitedFiles", "ProgressMeter"])
  Pkg.precompile()
  using CUDA
  println("CUDA functional: ", CUDA.functional())
'

## 5. Roda o treinamento
`main.jl` já chama `Roda()` no final e grava tudo em `Resultados/`. A flag `GKSwstype=100` garante a geração dos PDFs sem display.

In [ ]:
%%bash
cd "$PROJ_DIR"
GKSwstype=100 julia --project=. main.jl

## 6. Baixa os resultados

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('Resultados', 'zip', os.path.join(PROJ_DIR, 'Resultados'))
files.download('Resultados.zip')